In [1]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
xml_file = "Base_Model_Arch_vs_Structural_Clashes.xml"
output_json = "clashes_normalized.json"
usable_output_json = "usable_clashes.json"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def text_or_none(node, path):
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else None


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_name_value_nodes(parent, path):
    out = {}
    for item in parent.findall(path):
        name = text_or_none(item, "name")
        value = text_or_none(item, "value")
        if name:
            out[name] = value
    return out


def extract_created_at(clashresult):
    date_node = clashresult.find("./createddate/date")
    if date_node is None:
        return None

    try:
        return (
            f"{int(date_node.attrib.get('year')):04d}-"
            f"{int(date_node.attrib.get('month')):02d}-"
            f"{int(date_node.attrib.get('day')):02d}T"
            f"{int(date_node.attrib.get('hour', 0)):02d}:"
            f"{int(date_node.attrib.get('minute', 0)):02d}:"
            f"{int(date_node.attrib.get('second', 0)):02d}"
        )
    except Exception:
        return None


def extract_clash_point(clashresult):
    pt = clashresult.find("./clashpoint/pos3f")
    if pt is None:
        return None

    return {
        "x": to_float(pt.attrib.get("x")),
        "y": to_float(pt.attrib.get("y")),
        "z": to_float(pt.attrib.get("z")),
    }


def extract_revit_global_id(attrs):
    candidate_keys = [
        "Revit UniqueId",
        "Revit Unique ID",
        "UniqueId",
        "Unique ID",
        "GlobalId",
        "Global ID",
        "IfcGUID",
        "Ifc GUID",
    ]
    for key in candidate_keys:
        if key in attrs and attrs[key]:
            return attrs[key]
    return None


# ------------------------------------------------------------
# CLASSIFICATION
# ------------------------------------------------------------
def normalize_object_category(obj):
    """
    Creates a practical category using itemName + itemType + layer.
    Tweak these rules as you encounter more object types.
    """
    meta = obj.get("clashMetadata", {})
    item_name = (meta.get("itemName") or "").lower()
    item_type = (meta.get("itemType") or "").lower()
    layer = (meta.get("layer") or "").lower()

    # Architecture
    if "gypsum" in item_name:
        return "architectural_wall"

    if "wall" in item_name and "concrete" not in item_name:
        return "architectural_wall"

    if "studio unit" in item_name:
        return "architectural_space_or_unit"

    if "finish floor" in item_type or ("floor" in item_name and "structural" not in item_type):
        return "architectural_floor"

    # Structure
    if "structural framing" in item_type:
        return "structural_framing"

    if "structural columns" in item_type:
        return "structural_column"

    if "metal deck" in item_name or "deck" in item_name:
        return "structural_deck"

    if "concrete" in item_name:
        return "structural_concrete"

    # Layer hints
    if "tos" in layer:
        return "structural_framing"

    # Generic fallback
    if "solid" in item_type:
        return "generic_solid"

    return "other"


def infer_discipline(obj):
    """
    Maps normalized category to a broader discipline.
    """
    category = obj.get("normalizedCategory")

    if category in {
        "architectural_wall",
        "architectural_space_or_unit",
        "architectural_floor",
    }:
        return "architecture"

    if category in {
        "structural_framing",
        "structural_column",
        "structural_deck",
        "structural_concrete",
    }:
        return "structure"

    return "unknown"


# ------------------------------------------------------------
# CORE PARSERS
# ------------------------------------------------------------
def parse_clash_object(clashobject):
    attrs = parse_name_value_nodes(clashobject, "./objectattribute")
    tags = parse_name_value_nodes(clashobject, "./smarttags/smarttag")

    base_obj = {
        "revitGlobalId": extract_revit_global_id(attrs),
        "elementId": attrs.get("Element ID"),
        "clashMetadata": {
            "layer": text_or_none(clashobject, "./layer"),
            "itemName": tags.get("Item Name"),
            "itemType": tags.get("Item Type"),
        },
        "rawAttributes": attrs,
        "rawSmartTags": tags,
    }

    base_obj["normalizedCategory"] = normalize_object_category(base_obj)
    base_obj["discipline"] = infer_discipline(base_obj)

    return base_obj


def parse_clash_result(clashresult):
    return {
        "clashName": clashresult.attrib.get("name"),
        "clashGuid": clashresult.attrib.get("guid"),
        "clashMetadata": {
            "status": clashresult.attrib.get("status"),
            "description": text_or_none(clashresult, "./description"),
            "resultStatus": text_or_none(clashresult, "./resultstatus"),
            "distance": to_float(clashresult.attrib.get("distance")),
            "href": clashresult.attrib.get("href"),
            "gridLocation": text_or_none(clashresult, "./gridlocation"),
            "createdAt": extract_created_at(clashresult),
            "clashPoint": extract_clash_point(clashresult),
        },
        "objects": [
            parse_clash_object(obj)
            for obj in clashresult.findall("./clashobjects/clashobject")
        ],
    }


def parse_clash_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    output = {
        "sourceFile": Path(xml_path).name,
        "sourcePath": str(Path(xml_path).resolve()),
        "units": root.attrib.get("units"),
        "tests": []
    }

    for clashtest in root.findall(".//clashtest"):
        summary = clashtest.find("./summary")

        test_data = {
            "testName": clashtest.attrib.get("name"),
            "testType": clashtest.attrib.get("test_type"),
            "testStatus": clashtest.attrib.get("status"),
            "tolerance": clashtest.attrib.get("tolerance"),
            "summary": {
                "total": int(summary.attrib["total"]) if summary is not None and summary.attrib.get("total") else None,
                "new": int(summary.attrib["new"]) if summary is not None and summary.attrib.get("new") else None,
                "active": int(summary.attrib["active"]) if summary is not None and summary.attrib.get("active") else None,
                "reviewed": int(summary.attrib["reviewed"]) if summary is not None and summary.attrib.get("reviewed") else None,
                "approved": int(summary.attrib["approved"]) if summary is not None and summary.attrib.get("approved") else None,
                "resolved": int(summary.attrib["resolved"]) if summary is not None and summary.attrib.get("resolved") else None,
            },
            "clashes": []
        }

        for clashresult in clashtest.findall("./clashresults/clashresult"):
            test_data["clashes"].append(parse_clash_result(clashresult))

        output["tests"].append(test_data)

    return output


# ------------------------------------------------------------
# FLATTEN TO USABLE JSON OBJECTS
# ------------------------------------------------------------
def build_usable_clash_objects(parsed_data):
    usable = []

    for test in parsed_data.get("tests", []):
        for clash in test.get("clashes", []):
            objects = clash.get("objects", [])

            usable.append({
                "testName": test.get("testName"),
                "testType": test.get("testType"),
                "testStatus": test.get("testStatus"),
                "clashName": clash.get("clashName"),
                "clashGuid": clash.get("clashGuid"),
                "clashMetadata": clash.get("clashMetadata"),
                "objects": objects,
                "objectA": objects[0] if len(objects) > 0 else None,
                "objectB": objects[1] if len(objects) > 1 else None,
            })

    return usable


# ------------------------------------------------------------
# FILTERS
# ------------------------------------------------------------
def pair_match_unordered(value_a, value_b, wanted_a=None, wanted_b=None):
    a = (value_a or "").lower()
    b = (value_b or "").lower()

    if wanted_a and wanted_b:
        return {a, b} == {wanted_a.lower(), wanted_b.lower()}
    elif wanted_a:
        return a == wanted_a.lower() or b == wanted_a.lower()

    return False


def filter_clashes_by_category_pair(usable_clashes, category_a=None, category_b=None):
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        a = objects[0].get("normalizedCategory")
        b = objects[1].get("normalizedCategory")

        if pair_match_unordered(a, b, category_a, category_b):
            results.append(clash)

    return results


def filter_clashes_by_discipline_pair(usable_clashes, discipline_a=None, discipline_b=None):
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        a = objects[0].get("discipline")
        b = objects[1].get("discipline")

        if pair_match_unordered(a, b, discipline_a, discipline_b):
            results.append(clash)

    return results


def filter_clashes(usable_clashes, category_a=None, category_b=None,
                   discipline_a=None, discipline_b=None):
    """
    Combined filter:
    - if category filters are provided, clash must match them
    - if discipline filters are provided, clash must match them
    """
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        cat_a = objects[0].get("normalizedCategory")
        cat_b = objects[1].get("normalizedCategory")
        dis_a = objects[0].get("discipline")
        dis_b = objects[1].get("discipline")

        category_ok = True
        discipline_ok = True

        if category_a or category_b:
            category_ok = pair_match_unordered(cat_a, cat_b, category_a, category_b)

        if discipline_a or discipline_b:
            discipline_ok = pair_match_unordered(dis_a, dis_b, discipline_a, discipline_b)

        if category_ok and discipline_ok:
            results.append(clash)

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    data = parse_clash_xml(xml_file)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    usable_clashes = build_usable_clash_objects(data)

    with open(usable_output_json, "w", encoding="utf-8") as f:
        json.dump(usable_clashes, f, indent=2, ensure_ascii=False)

    print("Using:", Path(xml_file).resolve())
    print("Done. Wrote:", Path(output_json).resolve())
    print("Done. Wrote:", Path(usable_output_json).resolve())
    print(f"Total usable clash objects: {len(usable_clashes)}")

    # --------------------------------------------------------
    # EXAMPLES
    # --------------------------------------------------------

    # 1) Architecture vs Structure clashes
    arch_vs_struct = filter_clashes_by_discipline_pair(
        usable_clashes,
        discipline_a="architecture",
        discipline_b="structure"
    )
    print("\nArchitecture vs Structure clashes:", len(arch_vs_struct))

    # 2) Architectural wall vs Structural framing
    wall_vs_frame = filter_clashes_by_category_pair(
        usable_clashes,
        category_a="architectural_wall",
        category_b="structural_framing"
    )
    print("Architectural wall vs Structural framing clashes:", len(wall_vs_frame))

    # 3) Architectural wall vs Structural concrete
    wall_vs_concrete = filter_clashes_by_category_pair(
        usable_clashes,
        category_a="architectural_wall",
        category_b="structural_concrete"
    )
    print("Architectural wall vs Structural concrete clashes:", len(wall_vs_concrete))

    # 4) Combined filtering
    combined = filter_clashes(
        usable_clashes,
        category_a="architectural_wall",
        category_b="structural_concrete",
        discipline_a="architecture",
        discipline_b="structure"
    )
    print("Combined filtered clashes:", len(combined))

    # Preview first few
    print("\nFirst 3 usable clash objects:")
    print(json.dumps(usable_clashes[:3], indent=2, ensure_ascii=False))

    print("\nFirst 3 architectural wall vs structural concrete clashes:")
    print(json.dumps(wall_vs_concrete[:3], indent=2, ensure_ascii=False))

Using: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/Base_Model_Arch_vs_Structural_Clashes.xml
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/clashes_normalized.json
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/usable_clashes.json
Total usable clash objects: 3987

Architecture vs Structure clashes: 2622
Architectural wall vs Structural framing clashes: 2098
Architectural wall vs Structural concrete clashes: 66
Combined filtered clashes: 66

First 3 usable clash objects:
[
  {
    "testName": "Arch vs Struct",
    "testType": "hard",
    "testStatus": "ok",
    "clashName": "Clash1",
    "clashGuid": "f99fba96-1ab8-4f60-bc31-407511924e0d",
    "clashMetadata": {
      "status": "active",
      "description": "Hard",
      "resultStatus": "Active",
      "distance": -4.216,
      "href": "Base Model_Arch vs Structural Clashes_files\\cd000001.jpg",
      "gridLocation": "B-6 : L3",
      "createdAt": "2026-04-03T09:28:56",
      "clashPoint": {
        "x": 417625.629,
  

In [2]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
xml_file = "Base_Model_Arch_vs_Structural_Clashes.xml"
output_json = "clashes_normalized.json"
usable_output_json = "usable_clashes.json"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def text_or_none(node, path):
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else None


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_name_value_nodes(parent, path):
    out = {}
    for item in parent.findall(path):
        name = text_or_none(item, "name")
        value = text_or_none(item, "value")
        if name:
            out[name] = value
    return out


def extract_created_at(clashresult):
    date_node = clashresult.find("./createddate/date")
    if date_node is None:
        return None

    try:
        return (
            f"{int(date_node.attrib.get('year')):04d}-"
            f"{int(date_node.attrib.get('month')):02d}-"
            f"{int(date_node.attrib.get('day')):02d}T"
            f"{int(date_node.attrib.get('hour', 0)):02d}:"
            f"{int(date_node.attrib.get('minute', 0)):02d}:"
            f"{int(date_node.attrib.get('second', 0)):02d}"
        )
    except Exception:
        return None


def extract_clash_point(clashresult):
    pt = clashresult.find("./clashpoint/pos3f")
    if pt is None:
        return None

    return {
        "x": to_float(pt.attrib.get("x")),
        "y": to_float(pt.attrib.get("y")),
        "z": to_float(pt.attrib.get("z")),
    }


def extract_revit_global_id(attrs):
    candidate_keys = [
        "Revit UniqueId",
        "Revit Unique ID",
        "UniqueId",
        "Unique ID",
        "GlobalId",
        "Global ID",
        "IfcGUID",
        "Ifc GUID",
    ]
    for key in candidate_keys:
        if key in attrs and attrs[key]:
            return attrs[key]
    return None


# ------------------------------------------------------------
# CLASSIFICATION
# ------------------------------------------------------------
def normalize_object_category(obj):
    meta = obj.get("clashMetadata", {})
    item_name = (meta.get("itemName") or "").lower()
    item_type = (meta.get("itemType") or "").lower()
    layer = (meta.get("layer") or "").lower()

    # Architecture
    if "gypsum" in item_name:
        return "architectural_wall"

    if "wall" in item_name and "concrete" not in item_name:
        return "architectural_wall"

    if "studio unit" in item_name:
        return "architectural_space_or_unit"

    if "finish floor" in item_type or ("floor" in item_name and "structural" not in item_type):
        return "architectural_floor"

    # Structure
    if "structural framing" in item_type:
        return "structural_framing"

    if "structural columns" in item_type:
        return "structural_column"

    if "metal deck" in item_name or "deck" in item_name:
        return "structural_deck"

    if "concrete" in item_name:
        return "structural_concrete"

    # Layer hints
    if "tos" in layer:
        return "structural_framing"

    if "solid" in item_type:
        return "generic_solid"

    return "other"


def infer_discipline(obj):
    category = obj.get("normalizedCategory")

    if category in {
        "architectural_wall",
        "architectural_space_or_unit",
        "architectural_floor",
    }:
        return "architecture"

    if category in {
        "structural_framing",
        "structural_column",
        "structural_deck",
        "structural_concrete",
    }:
        return "structure"

    return "unknown"


# ------------------------------------------------------------
# SEVERITY + OWNER HEURISTICS
# ------------------------------------------------------------
def pair_key(obj_a, obj_b):
    a = obj_a.get("normalizedCategory", "other")
    b = obj_b.get("normalizedCategory", "other")
    return tuple(sorted([a, b]))


def compute_severity(clash, objects):
    """
    Returns one of:
    low / medium / high / critical

    Heuristic inputs:
    - absolute clash distance
    - category pair
    """
    distance = abs(clash.get("clashMetadata", {}).get("distance") or 0.0)

    categories = [obj.get("normalizedCategory", "other") for obj in objects[:2]]
    pair = tuple(sorted(categories))

    # Base severity from penetration distance
    # Adjust thresholds later if your report units/behavior differ.
    if distance >= 1.0:
        base = 4
    elif distance >= 0.30:
        base = 3
    elif distance >= 0.05:
        base = 2
    else:
        base = 1

    # Raise severity for more serious structural pairs
    severe_pairs = {
        ("architectural_floor", "structural_column"),
        ("architectural_floor", "structural_concrete"),
        ("architectural_floor", "structural_framing"),
        ("architectural_wall", "structural_column"),
        ("architectural_wall", "structural_concrete"),
        ("structural_column", "structural_deck"),
        ("structural_column", "structural_framing"),
    }

    moderate_pairs = {
        ("architectural_wall", "structural_deck"),
        ("architectural_wall", "structural_framing"),
        ("architectural_space_or_unit", "structural_deck"),
        ("architectural_space_or_unit", "structural_framing"),
        ("architectural_space_or_unit", "structural_concrete"),
    }

    if pair in severe_pairs:
        base += 1
    elif pair in moderate_pairs:
        base += 0.5

    # Clamp to 1..4
    if base >= 4:
        return "critical"
    elif base >= 3:
        return "high"
    elif base >= 2:
        return "medium"
    return "low"


def compute_recommended_owner(objects):
    """
    Returns:
    architecture / structure / shared / unknown

    Practical rule:
    - finish/partition/space planning issues usually start with architecture
    - framing/column/concrete geometry issues usually start with structure
    - some hard structural-vs-structural or ambiguous cases become shared
    """
    if len(objects) < 2:
        return "unknown"

    a = objects[0]
    b = objects[1]

    cat_a = a.get("normalizedCategory")
    cat_b = b.get("normalizedCategory")
    dis_a = a.get("discipline")
    dis_b = b.get("discipline")

    cats = {cat_a, cat_b}
    dis = {dis_a, dis_b}

    # Mixed architecture + structure
    if dis == {"architecture", "structure"}:
        # Architecture-led cases
        if "architectural_wall" in cats or "architectural_space_or_unit" in cats:
            return "architecture"

        if "architectural_floor" in cats:
            # floor clashes often need both, but structure usually needs to confirm geometry
            return "shared"

        return "shared"

    # Structure-only cases
    if dis == {"structure"}:
        return "structure"

    # Architecture-only cases
    if dis == {"architecture"}:
        return "architecture"

    return "shared"


def compute_recommended_owner_reason(objects):
    if len(objects) < 2:
        return "Insufficient object information to infer likely owner."

    a = objects[0]
    b = objects[1]

    cat_a = a.get("normalizedCategory")
    cat_b = b.get("normalizedCategory")
    dis_a = a.get("discipline")
    dis_b = b.get("discipline")

    if {dis_a, dis_b} == {"architecture", "structure"}:
        if "architectural_wall" in {cat_a, cat_b}:
            return "Architectural wall or partition clashes are usually first reviewed by architecture, then coordinated with structure."
        if "architectural_space_or_unit" in {cat_a, cat_b}:
            return "Space or unit envelope conflicts are usually first reviewed by architecture, then coordinated with structure."
        if "architectural_floor" in {cat_a, cat_b}:
            return "Floor-level conflicts often require both architectural intent and structural confirmation."
        return "Mixed-discipline clash; shared review is recommended."

    if dis_a == dis_b == "structure":
        return "Both clashing objects are structural, so structure is the likely first reviewer."

    if dis_a == dis_b == "architecture":
        return "Both clashing objects are architectural, so architecture is the likely first reviewer."

    return "Ownership is ambiguous from the current export, so shared review is recommended."


# ------------------------------------------------------------
# CORE PARSERS
# ------------------------------------------------------------
def parse_clash_object(clashobject):
    attrs = parse_name_value_nodes(clashobject, "./objectattribute")
    tags = parse_name_value_nodes(clashobject, "./smarttags/smarttag")

    base_obj = {
        "revitGlobalId": extract_revit_global_id(attrs),
        "elementId": attrs.get("Element ID"),
        "clashMetadata": {
            "layer": text_or_none(clashobject, "./layer"),
            "itemName": tags.get("Item Name"),
            "itemType": tags.get("Item Type"),
        },
        "rawAttributes": attrs,
        "rawSmartTags": tags,
    }

    base_obj["normalizedCategory"] = normalize_object_category(base_obj)
    base_obj["discipline"] = infer_discipline(base_obj)

    return base_obj


def parse_clash_result(clashresult):
    return {
        "clashName": clashresult.attrib.get("name"),
        "clashGuid": clashresult.attrib.get("guid"),
        "clashMetadata": {
            "status": clashresult.attrib.get("status"),
            "description": text_or_none(clashresult, "./description"),
            "resultStatus": text_or_none(clashresult, "./resultstatus"),
            "distance": to_float(clashresult.attrib.get("distance")),
            "href": clashresult.attrib.get("href"),
            "gridLocation": text_or_none(clashresult, "./gridlocation"),
            "createdAt": extract_created_at(clashresult),
            "clashPoint": extract_clash_point(clashresult),
        },
        "objects": [
            parse_clash_object(obj)
            for obj in clashresult.findall("./clashobjects/clashobject")
        ],
    }


def parse_clash_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    output = {
        "sourceFile": Path(xml_path).name,
        "sourcePath": str(Path(xml_path).resolve()),
        "units": root.attrib.get("units"),
        "tests": []
    }

    for clashtest in root.findall(".//clashtest"):
        summary = clashtest.find("./summary")

        test_data = {
            "testName": clashtest.attrib.get("name"),
            "testType": clashtest.attrib.get("test_type"),
            "testStatus": clashtest.attrib.get("status"),
            "tolerance": clashtest.attrib.get("tolerance"),
            "summary": {
                "total": int(summary.attrib["total"]) if summary is not None and summary.attrib.get("total") else None,
                "new": int(summary.attrib["new"]) if summary is not None and summary.attrib.get("new") else None,
                "active": int(summary.attrib["active"]) if summary is not None and summary.attrib.get("active") else None,
                "reviewed": int(summary.attrib["reviewed"]) if summary is not None and summary.attrib.get("reviewed") else None,
                "approved": int(summary.attrib["approved"]) if summary is not None and summary.attrib.get("approved") else None,
                "resolved": int(summary.attrib["resolved"]) if summary is not None and summary.attrib.get("resolved") else None,
            },
            "clashes": []
        }

        for clashresult in clashtest.findall("./clashresults/clashresult"):
            test_data["clashes"].append(parse_clash_result(clashresult))

        output["tests"].append(test_data)

    return output


# ------------------------------------------------------------
# FLATTEN TO USABLE JSON OBJECTS
# ------------------------------------------------------------
def build_usable_clash_objects(parsed_data):
    usable = []

    for test in parsed_data.get("tests", []):
        for clash in test.get("clashes", []):
            objects = clash.get("objects", [])

            severity = compute_severity(clash, objects)
            recommended_owner = compute_recommended_owner(objects)
            recommended_owner_reason = compute_recommended_owner_reason(objects)

            usable.append({
                "testName": test.get("testName"),
                "testType": test.get("testType"),
                "testStatus": test.get("testStatus"),
                "clashName": clash.get("clashName"),
                "clashGuid": clash.get("clashGuid"),
                "clashMetadata": clash.get("clashMetadata"),
                "severity": severity,
                "recommendedOwner": recommended_owner,
                "recommendedOwnerReason": recommended_owner_reason,
                "objects": objects,
                "objectA": objects[0] if len(objects) > 0 else None,
                "objectB": objects[1] if len(objects) > 1 else None,
            })

    return usable


# ------------------------------------------------------------
# FILTERS
# ------------------------------------------------------------
def pair_match_unordered(value_a, value_b, wanted_a=None, wanted_b=None):
    a = (value_a or "").lower()
    b = (value_b or "").lower()

    if wanted_a and wanted_b:
        return {a, b} == {wanted_a.lower(), wanted_b.lower()}
    elif wanted_a:
        return a == wanted_a.lower() or b == wanted_a.lower()

    return False


def filter_clashes_by_category_pair(usable_clashes, category_a=None, category_b=None):
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        a = objects[0].get("normalizedCategory")
        b = objects[1].get("normalizedCategory")

        if pair_match_unordered(a, b, category_a, category_b):
            results.append(clash)

    return results


def filter_clashes_by_discipline_pair(usable_clashes, discipline_a=None, discipline_b=None):
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        a = objects[0].get("discipline")
        b = objects[1].get("discipline")

        if pair_match_unordered(a, b, discipline_a, discipline_b):
            results.append(clash)

    return results


def filter_clashes(usable_clashes, category_a=None, category_b=None,
                   discipline_a=None, discipline_b=None,
                   severity=None, recommended_owner=None):
    results = []

    for clash in usable_clashes:
        objects = clash.get("objects", [])
        if len(objects) < 2:
            continue

        cat_a = objects[0].get("normalizedCategory")
        cat_b = objects[1].get("normalizedCategory")
        dis_a = objects[0].get("discipline")
        dis_b = objects[1].get("discipline")

        category_ok = True
        discipline_ok = True
        severity_ok = True
        owner_ok = True

        if category_a or category_b:
            category_ok = pair_match_unordered(cat_a, cat_b, category_a, category_b)

        if discipline_a or discipline_b:
            discipline_ok = pair_match_unordered(dis_a, dis_b, discipline_a, discipline_b)

        if severity:
            severity_ok = (clash.get("severity") or "").lower() == severity.lower()

        if recommended_owner:
            owner_ok = (clash.get("recommendedOwner") or "").lower() == recommended_owner.lower()

        if category_ok and discipline_ok and severity_ok and owner_ok:
            results.append(clash)

    return results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
if __name__ == "__main__":
    data = parse_clash_xml(xml_file)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    usable_clashes = build_usable_clash_objects(data)

    with open(usable_output_json, "w", encoding="utf-8") as f:
        json.dump(usable_clashes, f, indent=2, ensure_ascii=False)

    print("Using:", Path(xml_file).resolve())
    print("Done. Wrote:", Path(output_json).resolve())
    print("Done. Wrote:", Path(usable_output_json).resolve())
    print(f"Total usable clash objects: {len(usable_clashes)}")

    # Example 1
    high_arch_vs_struct = filter_clashes(
        usable_clashes,
        discipline_a="architecture",
        discipline_b="structure",
        severity="high"
    )
    print("\nHigh Architecture vs Structure clashes:", len(high_arch_vs_struct))

    # Example 2
    critical_wall_vs_concrete = filter_clashes(
        usable_clashes,
        category_a="architectural_wall",
        category_b="structural_concrete",
        severity="critical"
    )
    print("Critical architectural wall vs structural concrete clashes:", len(critical_wall_vs_concrete))

    # Example 3
    arch_owned = filter_clashes(
        usable_clashes,
        recommended_owner="architecture"
    )
    print("Architecture-owned clashes:", len(arch_owned))

    print("\nFirst 3 usable clash objects:")
    print(json.dumps(usable_clashes[:3], indent=2, ensure_ascii=False))

Using: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/Base_Model_Arch_vs_Structural_Clashes.xml
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/clashes_normalized.json
Done. Wrote: /home/b6b335a4-804b-48f8-a734-520d49ff1e55/usable_clashes.json
Total usable clash objects: 3987

High Architecture vs Structure clashes: 349
Critical architectural wall vs structural concrete clashes: 18
Architecture-owned clashes: 2735

First 3 usable clash objects:
[
  {
    "testName": "Arch vs Struct",
    "testType": "hard",
    "testStatus": "ok",
    "clashName": "Clash1",
    "clashGuid": "f99fba96-1ab8-4f60-bc31-407511924e0d",
    "clashMetadata": {
      "status": "active",
      "description": "Hard",
      "resultStatus": "Active",
      "distance": -4.216,
      "href": "Base Model_Arch vs Structural Clashes_files\\cd000001.jpg",
      "gridLocation": "B-6 : L3",
      "createdAt": "2026-04-03T09:28:56",
      "clashPoint": {
        "x": 417625.629,
        "y": 78709.244,
        "z": 247.